# 📊 Análise Exploratória de Dados
Neste notebook, vamos ligar o nosso motor de processamento (Spark) para ler os dados da empresa e responder a 5 perguntas sobre as Vendas e Avaliações da nossa equipe.

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, avg as _avg, unix_timestamp, round

# 1. LIGANDO o SparkSession...
spark = SparkSession.builder.appName("Analise_Exploratoria_Notebook").getOrCreate()

# 2. ler as nossas planilhas...
df_aval = spark.read.csv("data/base_avaliacoes.csv", header=True, inferSchema=True)
df_pessoas = spark.read.csv("data/base_pessoas.csv", header=True, inferSchema=True)
df_tel = spark.read.csv("data/base_telefonia.csv", header=True, inferSchema=True)

print("Ligado e dados carregados com sucesso!")

Ligado e dados carregados com sucesso!


## Análise de Vendas
Aqui vamos descobrir quem são os nossos melhores vendedores e qual é o "Ticket Médio" (o valor médio de cada venda).
Se a chamada foi uma "Reclamação", não entra na conta...

In [6]:
# Filtramos apenas as chamadas onde o Motivo é "Venda" E o valor não está vazio.
df_vendas_reais = df_tel.filter((col("Motivo") == "Venda") & (col("Valor venda").isNotNull()))

# --- TOP 5 VENDEDORES ---
top_vendedores = df_vendas_reais.groupBy("Username") \
    .agg(round(_sum("Valor venda"), 2).alias("Total_Vendido")) \
    .join(df_pessoas.select("Username", "Nome"), on="Username", how="inner") \
    .orderBy(col("Total_Vendido").desc()) \
    .limit(5)

print("Top 5 Vendedores:")
top_vendedores.select("Nome", "Total_Vendido").show(truncate=False)

# --- TICKET MÉDIO ---
ticket_medio = df_vendas_reais.groupBy("Username") \
    .agg(round(_avg("Valor venda"), 2).alias("Ticket_Medio")) \
    .join(df_pessoas.select("Username", "Nome"), on="Username", how="inner") \
    .orderBy(col("Ticket_Medio").desc())

print("Ticket Médio (Top 5):")
ticket_medio.select("Nome", "Ticket_Medio").show(5, truncate=False)

Top 5 Vendedores:
+-------------------------+-------------+
|Nome                     |Total_Vendido|
+-------------------------+-------------+
|Horácio Luísa Silveira   |9578.18      |
|Roldão Nataniel Daniel   |9534.16      |
|Nico Fausto Capela       |9353.7       |
|Valéria Cristiana Ventura|9225.61      |
|Brígida Gui Rocha        |9175.45      |
+-------------------------+-------------+

Ticket Médio (Top 5):
+-------------------------+------------+
|Nome                     |Ticket_Medio|
+-------------------------+------------+
|Bartolomeu Alexandre Vale|65.79       |
|Roldão Nataniel Daniel   |65.3        |
|Custódia Rosário Duarte  |65.1        |
|Romeu Alberta Mateus     |64.56       |
|Valéria Cristiana Ventura|64.51       |
+-------------------------+------------+
only showing top 5 rows



## Tempo Médio das Ligações
**OBS** chamadas com "tempo negativo" são erros do sistema e serão ignoradas para não estragar a nossa média!!!

In [7]:
# Transformamos os textos de data em "segundos"...
df_tempo = df_tel.withColumn("inicio_ts", unix_timestamp("inicio_ligacao")) \
                 .withColumn("fim_ts", unix_timestamp("fim_ligação")) \
                 .withColumn("duracao_minutos", (col("fim_ts") - col("inicio_ts")) / 60.0)

# Calculamos a média ignorando os valores menores que zero...
tempo_medio = df_tempo.filter(col("duracao_minutos") >= 0) \
                      .select(round(_avg("duracao_minutos"), 2).alias("Tempo_Medio_Minutos"))

print(" Tempo médio das ligações :")
tempo_medio.show()

 Tempo médio das ligações :
+-------------------+
|Tempo_Medio_Minutos|
+-------------------+
|               9.92|
+-------------------+



## Análise de Avaliações
Vamos entender como os clientes estão a avaliar os nossos atendimentos.

In [8]:
# --- NOTA MÉDIA GERAL ---
nota_geral = df_aval.select(round(_avg("Nota"), 2).alias("Nota_Media_Geral"))
print("Nota média geral:")
nota_geral.show()

# --- PIOR VENDEDOR AVALIADO ---
pior_vendedor = df_aval.groupBy("Username") \
    .agg(round(_avg("Nota"), 2).alias("Media_Avaliacao")) \
    .join(df_pessoas.select("Username", "Nome"), on="Username", how="inner") \
    .orderBy(col("Media_Avaliacao").asc()) \
    .limit(1)

print("Pior vendedor avaliado:")
pior_vendedor.select("Nome", "Media_Avaliacao").show(truncate=False)

Nota média geral:
+----------------+
|Nota_Media_Geral|
+----------------+
|             7.5|
+----------------+

Pior vendedor avaliado:
+-------------------------+---------------+
|Nome                     |Media_Avaliacao|
+-------------------------+---------------+
|Brunilda Assunção Santana|5.95           |
+-------------------------+---------------+

